# Chapter 2 Lab — Mapping the Persuadability Spectrum

**Basal Cognition** · Dr. Ernesto Lee · [BasalCognition.com](https://basalcognition.com)

---

In this lab you will run the four-question diagnostic procedure from Chapter 2 on a real AI system. You will probe an LLM API with:

1. **Identical inputs** — does it produce different outputs? (Tier 1 test)
2. **State persistence** — does anything carry over between fresh sessions? (Tier 2 test)
3. **Novel generalization** — can it apply knowledge to scenarios it likely hasn't seen? (Tier 3 test)
4. **Goal reasoning** — can it explain *why* it is doing something and catch conflicting instructions? (Tier 4 test)

At the end you will score the system and plot its estimated position on the persuadability spectrum.

**Time:** 60–90 minutes  
**Prerequisite:** A free OpenAI or OpenRouter API key

In [ ]:
# Install dependencies
!pip install openai matplotlib numpy pandas --quiet

In [ ]:
import openai
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import json
import time
from typing import List, Dict

# ── Configuration ──────────────────────────────────────────────────────────────
# Paste your API key below, OR set it as a Colab secret named OPENAI_API_KEY
import os
try:
    from google.colab import userdata
    API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_API_KEY_HERE')

MODEL = 'gpt-4o-mini'  # change to any model you have access to

client = openai.OpenAI(api_key=API_KEY)
print(f"Using model: {MODEL}")

## Test 1 — Tier 1 Check: Does it produce different outputs for the same input?

A Tier 1 mechanism produces identical output for identical input every time. A system with any stochasticity is already above Tier 1.

We will send the exact same prompt 5 times with `temperature=0` and 5 times with `temperature=1` and measure the variance.

In [ ]:
TIER1_PROMPT = "Describe a thermostat in exactly one sentence."

def query(prompt: str, system: str = "", temp: float = 0.7, n: int = 1) -> List[str]:
    """Send a prompt and return n responses."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    
    responses = []
    for _ in range(n):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temp,
            max_tokens=200
        )
        responses.append(resp.choices[0].message.content.strip())
        time.sleep(0.5)  # rate-limit courtesy
    return responses

print("Running Tier 1 test — 5 calls at temperature=0, 5 at temperature=1 ...")
low_temp_responses = query(TIER1_PROMPT, temp=0.0, n=5)
high_temp_responses = query(TIER1_PROMPT, temp=1.0, n=5)

print("\n--- Low temperature (0.0) ---")
for i, r in enumerate(low_temp_responses, 1):
    print(f"[{i}] {r}")

print("\n--- High temperature (1.0) ---")
for i, r in enumerate(high_temp_responses, 1):
    print(f"[{i}] {r}")

unique_low = len(set(low_temp_responses))
unique_high = len(set(high_temp_responses))
print(f"\nUnique responses at temp=0: {unique_low}/5")
print(f"Unique responses at temp=1: {unique_high}/5")

tier1_score = 1 if unique_high > 1 else 0  # 1 = above Tier 1
print(f"\nTier 1 check: {'PASS — system has variability (above Tier 1)' if tier1_score else 'FAIL — pure mechanism'}")

## Test 2 — Tier 2 Check: Does state persist across fresh sessions?

We tell the model a fact in one call, start a completely fresh client object, and ask for the fact back. A stateless reactive system (Tier 2) will not remember it.

In [ ]:
# TODO: Implement Test 2
# Step 1: Tell the model your favorite color in one API call
# Step 2: Create a brand new client object (fresh session, no prior messages)
# Step 3: Ask "what is my favorite color?" with no context
# Step 4: Record and print the response
# Step 5: Score: does the model know? If not, confirm Tier 2 statelessness.

# YOUR CODE HERE
print("TODO: implement Test 2")
print("Hint: use a fresh openai.OpenAI() instance and send messages WITHOUT the prior conversation.")

# tier2_score = 0  # 0 = no state persistence (Tier 2), 1 = has state (Tier 3+)
tier2_score = 0  # default assumption; update after your test

## Test 3 — Tier 3 Check: Novel generalization

Give the model a scenario it almost certainly hasn't seen verbatim. A Tier 3 system can reason by analogy and produce a sensible, novel response. A Tier 2 system pattern-matches but fails to generalize coherently.

In [ ]:
NOVEL_SCENARIO = """
A species of deep-sea jellyfish has been discovered that can solve simple mazes 
by adjusting the electrical charge on its tentacles. The jellyfish has no brain 
and no neurons. Where would you place it on the persuadability spectrum — 
Tier 1, 2, 3, or 4? Explain your reasoning in 3 sentences.
"""

print("Running generalization test ...")
responses = query(NOVEL_SCENARIO, n=3)
for i, r in enumerate(responses, 1):
    print(f"\n[Response {i}]\n{r}")

# TODO: Evaluate the responses manually.
# Does the model apply the tier framework correctly to a scenario it hasn't memorized?
# Does it give a consistent answer across the 3 responses?
# Score 1 if the model reasons coherently; score 0 if it just pattern-matches surface words.

tier3_score = None  # replace with 0 or 1 after reviewing responses
print("\nSet tier3_score to 0 or 1 based on your evaluation above.")

## Test 4 — Tier 4 Check: Goal reasoning and conflict detection

We give the model a system prompt with a clear goal, then a user instruction that conflicts with it. A Tier 4 system notices the conflict and flags it. A Tier 3 system follows whichever instruction is more recent.

In [ ]:
SYSTEM_GOAL = "Your goal is to help the user reduce their carbon footprint. Recommend only actions that lower emissions."
CONFLICTING_USER = "I want to take a long-haul flight to a climate summit to learn about sustainability. Should I go?"

print("Running goal-conflict test ...")
responses = query(CONFLICTING_USER, system=SYSTEM_GOAL, temp=0.7, n=2)
for i, r in enumerate(responses, 1):
    print(f"\n[Response {i}]\n{r}")

# TODO: Evaluate:
# Does the model notice the tension between the stated goal (reduce emissions)
# and the user request (take a high-emission flight)?
# Does it flag the conflict, or does it just answer the question?
# Does it explain WHY there is tension?

tier4_score = None  # replace with 0 or 1
print("\nSet tier4_score to 0 or 1 based on your evaluation above.")

## Plotting the Result

Run this cell after setting all four scores. Your plot will show where the model lands on the persuadability spectrum.

In [ ]:
# Replace None values with your scores before running
scores = {
    'Tier 1 (variability)': tier1_score,
    'Tier 2 (stateless)': tier2_score,
    'Tier 3 (generalize)': tier3_score,
    'Tier 4 (goal-reason)': tier4_score,
}

if None in scores.values():
    print("⚠ Set all four scores (0 or 1) before plotting.")
else:
    total = sum(scores.values())
    position = total / 4.0  # 0.0 = Tier 1, 1.0 = Tier 4
    
    fig, ax = plt.subplots(figsize=(12, 3))
    
    # Draw the spectrum bar
    gradient = np.linspace(0, 1, 256).reshape(1, -1)
    ax.imshow(gradient, extent=[0, 1, 0, 1], aspect='auto', cmap='RdYlGn', alpha=0.4)
    
    # Tier labels
    tier_positions = [0.1, 0.35, 0.65, 0.9]
    tier_labels = ['Tier 1\nMechanism', 'Tier 2\nReactive', 'Tier 3\nTrainable', 'Tier 4\nPersuadable']
    for pos, label in zip(tier_positions, tier_labels):
        ax.text(pos, 0.5, label, ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    # Model position
    ax.axvline(x=position, color='#006080', linewidth=3, linestyle='--')
    ax.text(position, 0.95, f'{MODEL}\nScore: {total}/4', ha='center', va='top',
            fontsize=11, color='#006080', fontweight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xlabel('Persuadability Spectrum →', fontsize=12)
    ax.set_title('Chapter 2 Lab: Model Tier Placement', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('ch02-lab-result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\nFinal score: {total}/4 → estimated position: {position:.2f} on the spectrum")

## Deliverable

Write a short paragraph (150–250 words) answering:

1. Where did your model score on the spectrum?
2. Which test was the most revealing? Why?
3. Based on your score, which intervention rung from the AI ladder would be most effective for improving this model's performance on your chosen task?
4. What would you need to test to move the score by one tier in either direction?

Paste your paragraph in the cell below.

**Your answer here:**

*(double-click to edit)*